# **Retreival part**

This part is dedicated to experience retreival part and analysing multiple approaches of chunking and searching for relevant documents.

## **Chunking**

In this part, we are going to do an experience on different chunking methods :

- Fixed length, overlap : basic chunking
- Chunking based on paragraphs

In [1]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from Dataset.upload import get_data
from typing import Literal
import pandas as pd

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cu130
CUDA build: 13.0
CUDA available: True


### **Basic Chunk (simple, paragraph)**

In [ ]:
def chunk_characters(text : str, overlap : int, chunk_size : int, chunk_type : Literal["recursive", "simple"] = "recursive") -> list : 
    
    if chunk_type == "simple" : 
        splitter = CharacterTextSplitter(
            separator="",
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )
    elif chunk_type == "recursive" : 
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap,
            length_function=len,
        )


    chunks = splitter.split_text(text)

    return chunks

In [4]:
cour_d_appel = await get_data(juridication="cour_d_appel")

In [5]:
tribunal_judiciaire = await get_data(juridication="tribunal_judiciaire", length=50)

In [6]:
cour_de_cassation = await get_data(juridication="cour_de_cassation")

### **Collection handling with chromaDB**

In [7]:
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [8]:
def clear_collection(collection: chromadb.Collection) -> None:
    """Remove existing records from a collection."""
    collection_count = collection.count()

    if collection_count > 0:
        all_ids = collection.get(include=[])["ids"]
        if all_ids:
           collection.delete(ids=all_ids)
        print(f"Cleared {collection_count} records from '{collection.name}'.")
    else:
        print(f"Collection '{collection.name}' is already empty.")

In [9]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=device,
)

cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
import re
DB_PATH = Path(".").resolve().parent / "Collections"
# # A multilingual baseline suitable for French text
# EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# # Load the model once, rather than once per upload
# embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

import unicodedata

def clean_text(text: str) -> str:
    # Normalize Unicode characters
    text = unicodedata.normalize("NFC", text)

    # Remove common page-break and control characters
    text = text.replace("\x0c", " ")
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # Remove spaces around hyphens used as line-wrap artifacts
    # (Avoid blindly joining words around every hyphen.)

    return text.strip()

print(f"Embedding model running on: {device}")

def create_documents(
    juridication : Literal["cours d'appel","tribunal judiciaire","cours de cassatioin"],
    data : pd.DataFrame,
    chunk_size: int = 300,
    overlap: int = 100,
    ) -> list :
    
    ids: list[str] = []
    documents: list[str] = []
    metadatas: list[dict] = []

    # Build records from the source DataFrame
    for row in data.itertuples(index=False):
        
        if pd.isna(row.text) or not str(row.text).strip():
            continue

        source_id = str(row.id)
        source_text = str(row.text)
        location = str(row.location)
        source_themes = row.themes
        if isinstance(source_themes, (list, tuple, set)):
            themes_text = ", ".join(map(str, source_themes))
        elif source_themes is None or (
            isinstance(source_themes, float) and pd.isna(source_themes)
        ):
            themes_text = ""
        else:
            themes_text = str(source_themes)

        text_to_chunk = f"Numéros de l'affaire : {row.numbers}; \n Text juridique : {source_text}; \n THEMES : {themes_text}; \n"
        clean_text_to_chunk = clean_text(text_to_chunk)
        chunks = chunk_characters(
            text=text_to_chunk,
            overlap=overlap,
            chunk_size=chunk_size,
        )

        for chunk_index, chunk in enumerate(chunks):
            chunk = str(chunk).strip()

            if not chunk:
                continue

            # Zero-based chunk indices
            chunk_id = f"{source_id}_chunk_{chunk_index}"
            numbers= ""

            for number in row.numbers :
                numbers += f"{str(number)},"
                
            ids.append(chunk_id)
            documents.append(chunk)
            metadatas.append(
                {
                    "source_id": source_id,
                    "chunk_index": chunk_index,
                    "themes": themes_text,
                    "juridication" : juridication,
                    "location" : location,
                    "Numéros de l'affaire" : numbers
                }
            )
    return documents, ids, metadatas

def embed_docs(documents : list, batch_size : int, ids : list, metadatas : list, collection : chromadb.Collection)-> bool:
    try : 
        for start in range(0, len(documents), batch_size):
                end = start + batch_size

                batch_documents = documents[start:end]
                batch_ids = ids[start:end]
                batch_metadatas = metadatas[start:end]

                embeddings = embedding_model.encode(
                    batch_documents,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    show_progress_bar=True,
                    batch_size=batch_size,
                )

                collection.add(
                    ids=batch_ids,
                    documents=batch_documents,
                    metadatas=batch_metadatas,
                    embeddings=embeddings.tolist(),
                )      
        print("Added to the collection")
        return True

    except Exception : 
        return False
    
def upload_to_collection(
    data_name : str,
    juridication : Literal["cours d'appel","tribunal judiciaire","cours de cassation"],
    data: pd.DataFrame,
    chunk_size: int = 300,
    overlap: int = 100,
    batch_size: int = 64,
) -> chromadb.Collection:
    """Chunk source documents and store them in a persistent ChromaDB collection."""

    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Require chunk_size > 0 and 0 <= overlap < chunk_size.")

    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    required_columns = {"id", "themes", "text"}
    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise ValueError(f"Missing DataFrame columns: {missing_columns}")

    collection_name = f"{data_name}"

    # Reuse one persistent database directory
    DB_PATH.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(DB_PATH))

    # Get or create the collection.
    # Embeddings are supplied explicitly, so no embedding_function is needed.
    collection = client.get_or_create_collection(
        name=collection_name,
        configuration={
            "hnsw": {"space": "cosine"}
        }
    )

    # This function intentionally rebuilds the collection on every upload.
    clear_collection(collection)

    documents, ids, metadatas = create_documents(data = data, chunk_size=chunk_size, overlap=overlap, juridication=juridication)

    if not documents:
        print("No non-empty chunks were generated.")
        return collection

    # Embed and upload records in batches
    result = embed_docs(documents=documents, batch_size=batch_size, ids=ids, metadatas=metadatas, collection=collection)

    if result : 
        print(
            f"Uploaded {collection.count()} chunks to '{collection.name}'."
        )

        return collection, collection_name


Embedding model running on: cuda


### **Search Functions**

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer
from rank_bm25 import BM25Okapi

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

In [12]:
def KeywordSearch_TFIDF(query : str, top_k : int, collection : chromadb.Collection) -> list :
    #Call the vectorizer
    vectorizer = TfidfVectorizer()

    stored = collection.get(
                include=["documents", "metadatas"]
            )
    
    docs = stored["documents"]
    ids = stored["ids"]
    metadatas = stored["metadatas"]
    
    document_vectors = vectorizer.fit_transform(docs)

    # Convert the query into the same TF-IDF space
    query_vector = vectorizer.transform([query])
    # Score the query against all indexed documents
    scores = cosine_similarity(
        query_vector,
        document_vectors
    ).ravel()

    # Sort from highest score to lowest
    ranked_indices = scores.argsort()[::-1]

    results = []

    for idx in ranked_indices:
        if scores[idx] <= 0:
            break

        results.append({
            "document": docs[idx],
            "score": float(scores[idx]),
            "id" : ids[idx],
            "metadatas" : metadatas[idx]
        })

        if len(results) == top_k:
            break

    return results
    
def KeywordSearch_BM25(query : str, top_k : int, collection : chromadb.Collection) -> list :

    stored = collection.get(
            include=["documents", "metadatas"]
        )
    docs = stored["documents"]
    ids = stored["ids"]
    metadatas = stored["metadatas"]
    
    tokenized_docs = [tokenizer.tokenize(doc) for doc in docs]
    bm25 = BM25Okapi(tokenized_docs)

    scores = bm25.get_scores(tokenizer.tokenize(query))

    ranked_indices = scores.argsort()[::-1]

    return [
        {
            "document": docs[i],
            "id" : ids[i],
            "score": float(scores[i]),
            "metadatas" : metadatas[i]
        }
        for i in ranked_indices[:top_k]
        if scores[i] > 0
    ]

def SemanticSearch(
    query: str,
    collection: chromadb.Collection,
    top_k: int = 3
) -> list[dict]:

    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for i, document in enumerate(results["documents"][0]):
        output.append({
            "document": document,
            "id" : results['ids'][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return output
    

# **Testing phase**

- All the collection are based on cosine similarity
- When we query the database, chromadb will simply use the cosine similarity to retreive the relevant chunks

In [17]:
collection_cour_d_appel, collection_name_cour_d_appel = upload_to_collection(data_name="cour_d_appel", data=cour_d_appel, juridication = "cours d'appel", chunk_size=1000, overlap=400)

Cleared 1921 records from 'cour_d_appel'.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Added to the collection
Uploaded 1921 chunks to 'cour_d_appel'.


In [18]:
client = chromadb.PersistentClient(path=DB_PATH)

client.delete_collection(name="tribunal_judiciaire")

In [20]:
collection_tribunal_judiciaire, collection_name_tribunal_judiciaire = upload_to_collection(data_name="tribunal_judiciaire", data=tribunal_judiciaire, juridication="tribunal judiciaire", chunk_size=1000, overlap=500)

Collection 'tribunal_judiciaire' is already empty.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Added to the collection
Uploaded 7553 chunks to 'tribunal_judiciaire'.


In [21]:
collection_cour_de_cassation, collection_name_cour_de_cassation = upload_to_collection(data_name="cour_de_cassation", data=cour_de_cassation, juridication="cours de cassation", chunk_size=1000, overlap=400)

Cleared 515 records from 'cour_de_cassation'.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Added to the collection
Uploaded 515 chunks to 'cour_de_cassation'.


In [41]:
bm25_res = KeywordSearch_BM25(query="contrat de travail", collection=collection_cour_d_appel, top_k=5)

In [42]:
bm25_res

[{'document': 'la rupture abusive de son contrat de travail en raison de la modification substantielle de ses conditions de travail.',
  'id': '6253c84abd3db21cbdd84ccf_chunk_17',
  'score': 15.034576737945322,
  'metadatas': {'chunk_index': 17,
   'themes': "contrat de travail, execution, modification, modification imposée par l'employeur",
   'juridication': "cours d'appel",
   'location': "Cour d'appel de Versailles",
   'source_id': '6253c84abd3db21cbdd84ccf',
   "Numéros de l'affaire": '1996-22443,'}},
 {'document': "incorporé au contrat de travail de Monsieur X... de sorte que la dénonciation de cet usage par l'employeur ne saurait constituer une modification substantielle du contrat de travail;",
  'id': '6253c84abd3db21cbdd84ccf_chunk_58',
  'score': 14.362586937006167,
  'metadatas': {"Numéros de l'affaire": '1996-22443,',
   'themes': "contrat de travail, execution, modification, modification imposée par l'employeur",
   'location': "Cour d'appel de Versailles",
   'chunk_ind

In [45]:
tfidf_res =KeywordSearch_TFIDF(query="contrat de travail", collection=collection_cour_d_appel, top_k=5)

In [46]:
tfidf_res

[{'document': 'la rupture abusive de son contrat de travail en raison de la modification substantielle de ses conditions de travail.',
  'score': 0.5879752865095359,
  'id': '6253c84abd3db21cbdd84ccf_chunk_17',
  'metadatas': {'chunk_index': 17,
   "Numéros de l'affaire": '1996-22443,',
   'juridication': "cours d'appel",
   'location': "Cour d'appel de Versailles",
   'source_id': '6253c84abd3db21cbdd84ccf',
   'themes': "contrat de travail, execution, modification, modification imposée par l'employeur"}},
 {'document': "incorporé au contrat de travail de Monsieur X... de sorte que la dénonciation de cet usage par l'employeur ne saurait constituer une modification substantielle du contrat de travail;",
  'score': 0.5258121743279984,
  'id': '6253c84abd3db21cbdd84ccf_chunk_58',
  'metadatas': {'source_id': '6253c84abd3db21cbdd84ccf',
   'location': "Cour d'appel de Versailles",
   'themes': "contrat de travail, execution, modification, modification imposée par l'employeur",
   'juridic

In [47]:
sem_res = SemanticSearch(query="contrat de travail", collection=collection_cour_d_appel, top_k=5)

In [48]:
sem_res

[{'document': 'S.M.I. SA aux frais de contredit. ARRET PRONONCE PAR MONSIEUR ASSIÉ, PRESIDENT ET ONT SIGNE LE PRESENT ARRET LE GREFFIER DIVISIONNAIRE LE PRESIDENT A. PECHE-MONTREUIL F. ASSIE;',
  'id': '6253c839bd3db21cbdd84a77_chunk_55',
  'metadata': {"Numéros de l'affaire": '1996-8123,',
   'source_id': '6253c839bd3db21cbdd84a77',
   'juridication': "cours d'appel",
   'location': "Cour d'appel de Versailles",
   'chunk_index': 55,
   'themes': 'conventions internationales, accords et conventions divers, convention de bruxelles du 27 septembre 1968'},
  'distance': 0.621893048286438},
 {'document': 'de Procédure Civile. ARRET PRONONCE PAR MONSIEUR ASSIÉ, PRESIDENT ET ONT SIGNE LE PRESENT ARRET LE GREFFIER DIVISIONNAIRE',
  'id': '6253c844bd3db21cbdd84be2_chunk_80',
  'metadata': {"Numéros de l'affaire": '1995-8297,',
   'chunk_index': 80,
   'themes': 'agent commercial, contrat, fin',
   'juridication': "cours d'appel",
   'location': "Cour d'appel de Versailles",
   'source_id': '6

### **Defining the metrics functions for the fusion (hybridSearch)**

RRF method

In [49]:
from collections import defaultdict

def reciprocal_rank_fusion(
    result_lists: list[list[str]],
    k: int = 60,
) -> list[tuple[str, float]]:

    scores = defaultdict(float)

    for results in result_lists:
        for rank, doc_id in enumerate(results, start=1):
            scores[doc_id] += (1 / (k + rank))*100

    return sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )


In [50]:
bm25_res_ids = [id["id"] for id in bm25_res]
tfidf_res_ids = [id["id"] for id in tfidf_res]
sem_res_ids = [id["id"] for id in sem_res]

list_1 =[bm25_res_ids, sem_res_ids] 
list_2 =[tfidf_res_ids, sem_res_ids] 
list_3 =[tfidf_res_ids, bm25_res_ids] 

In [51]:
res1 = reciprocal_rank_fusion(result_lists=list_1)
for rec in res1: 
    print(rec)
print(f"len : {len(res1)}")

('6253c84abd3db21cbdd84ccf_chunk_17', 1.639344262295082)
('6253c839bd3db21cbdd84a77_chunk_55', 1.639344262295082)
('6253c84abd3db21cbdd84ccf_chunk_58', 1.6129032258064515)
('6253c844bd3db21cbdd84be2_chunk_80', 1.6129032258064515)
('6253c845bd3db21cbdd84c2f_chunk_88', 1.5873015873015872)
('6253c83abd3db21cbdd84a7b_chunk_65', 1.5873015873015872)
('6253c84abd3db21cbdd84ccf_chunk_50', 1.5625)
('6253c844bd3db21cbdd84be7_chunk_43', 1.5625)
('6253c84abd3db21cbdd84cce_chunk_27', 1.5384615384615385)
('6253c84cbd3db21cbdd84d28_chunk_41', 1.5384615384615385)
len : 10


In [52]:
res2  = reciprocal_rank_fusion(result_lists = list_2)
for rec in res2: 
    print(rec)
print(f"len : {len(res2)}")

('6253c84abd3db21cbdd84ccf_chunk_17', 1.639344262295082)
('6253c839bd3db21cbdd84a77_chunk_55', 1.639344262295082)
('6253c84abd3db21cbdd84ccf_chunk_58', 1.6129032258064515)
('6253c844bd3db21cbdd84be2_chunk_80', 1.6129032258064515)
('6253c845bd3db21cbdd84c2f_chunk_88', 1.5873015873015872)
('6253c83abd3db21cbdd84a7b_chunk_65', 1.5873015873015872)
('6253c84abd3db21cbdd84cd0_chunk_22', 1.5625)
('6253c844bd3db21cbdd84be7_chunk_43', 1.5625)
('6253c84abd3db21cbdd84ccf_chunk_50', 1.5384615384615385)
('6253c84cbd3db21cbdd84d28_chunk_41', 1.5384615384615385)
len : 10


In [53]:
res3 = reciprocal_rank_fusion(result_lists = list_3)
for rec in res3: 
    print(rec)
print(f"len : {len(res3)}")

('6253c84abd3db21cbdd84ccf_chunk_17', 3.278688524590164)
('6253c84abd3db21cbdd84ccf_chunk_58', 3.225806451612903)
('6253c845bd3db21cbdd84c2f_chunk_88', 3.1746031746031744)
('6253c84abd3db21cbdd84ccf_chunk_50', 3.1009615384615383)
('6253c84abd3db21cbdd84cd0_chunk_22', 1.5625)
('6253c84abd3db21cbdd84cce_chunk_27', 1.5384615384615385)
len : 6


In [62]:
ids = [t for t,n in res3]
list4 = [ids, sem_res_ids]
res4 = reciprocal_rank_fusion(list4)

In [65]:
chunks = res4[:5]

In [70]:
chunks = [t for t, n in chunks]

In [71]:
chunks

['6253c84abd3db21cbdd84ccf_chunk_17',
 '6253c839bd3db21cbdd84a77_chunk_55',
 '6253c84abd3db21cbdd84ccf_chunk_58',
 '6253c844bd3db21cbdd84be2_chunk_80',
 '6253c845bd3db21cbdd84c2f_chunk_88']

Reranking

ModuleNotFoundError: No module named 'Backend'

Weighted score combination